In [46]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder,OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import re
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df=pd.read_csv(f"/content/bengaluru_house_prices.csv")

In [3]:
df

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00
...,...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5 Bedroom,ArsiaEx,3453,4.0,0.0,231.00
13316,Super built-up Area,Ready To Move,Richards Town,4 BHK,NaN,3600,5.0,NaN,400.00
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2 BHK,Mahla T,1141,2.0,1.0,60.00
13318,Super built-up Area,18-Jun,Padmanabhanagar,4 BHK,SollyCl,4689,4.0,1.0,488.00


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


In [5]:
df.isnull().sum()


,0
area_type,0
availability,0
location,1
size,16
society,5502
total_sqft,0
bath,73
balcony,609
price,0


In [6]:
df.value_counts("total_sqft")

,count
total_sqft,
1200,843
1100,221
1500,205
2400,196
600,180
...,...
981 - 1249,1
981,1
1005.03 - 1252.49,1


# Cleaning data

In [9]:
def convert_sqft(x):
    """Handles plain numbers, ranges like '2100-2850', and unit strings like '34.46Sq. Meter'."""
    x = str(x).strip()
    if "-" in x:
        parts = x.split("-")
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return np.nan
    try:
        return float(x)
    except ValueError:
        # strip any trailing unit text, keep leading numeric part
        import re
        match = re.match(r"([\d.]+)", x)
        return float(match.group(1)) if match else np.nan

In [11]:
df["total_sqft"] = df["total_sqft"].apply(convert_sqft)

In [12]:
df

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056.0,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600.0,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440.0,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521.0,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200.0,2.0,1.0,51.00
...,...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5 Bedroom,ArsiaEx,3453.0,4.0,0.0,231.00
13316,Super built-up Area,Ready To Move,Richards Town,4 BHK,NaN,3600.0,5.0,NaN,400.00
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2 BHK,Mahla T,1141.0,2.0,1.0,60.00
13318,Super built-up Area,18-Jun,Padmanabhanagar,4 BHK,SollyCl,4689.0,4.0,1.0,488.00


In [13]:
df["bhk"] = df["size"].str.extract(r"(\d+)").astype(float)

In [14]:
# Drop columns that are mostly noise
df = df.drop(columns=["society", "size"])


In [15]:
df["availability"] = df["availability"].apply(
    lambda x: 1 if str(x).strip() == "Ready To Move" else 0
)
df = df.rename(columns={"availability": "ready_to_move"})

In [16]:
df = df.dropna(subset=["location", "total_sqft", "price"])

In [17]:
df["bath"] = df["bath"].fillna(df["bath"].median())
df["balcony"] = df["balcony"].fillna(df["balcony"].median())
df["bhk"] = df["bhk"].fillna(df["bhk"].median())

In [18]:
df.shape

(13319, 8)

# Data preprocessing

In [19]:
df["sqft_per_bhk"] = df["total_sqft"] / df["bhk"]

In [20]:
df = pd.get_dummies(df, columns=["location", "area_type"], drop_first=True)

In [21]:
X = df.drop(columns=["price"])
y = df["price"]

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (10655, 1313) Test shape: (2664, 1313)


In [23]:
numeric_cols = ["total_sqft", "bath", "balcony", "bhk", "sqft_per_bhk", "ready_to_move"]

In [24]:
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Model

In [37]:
n_features = X_train.shape[1]

In [38]:
model = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)  # linear output for regression (price in lakhs)
])

In [39]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 128)            │       168,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 178,561 (697.50 KB)

 Trainable params: 178,561 (697.50 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True
)
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=8, min_lr=1e-6
)


In [41]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=2
)

Epoch 1/300
267/267 - 3s - 12ms/step - loss: 21550.8965 - mae: 67.1415 - val_loss: 16471.3086 - val_mae: 43.3881 - learning_rate: 0.0010
Epoch 2/300
267/267 - 2s - 6ms/step - loss: 13646.3379 - mae: 42.5254 - val_loss: 20208.2070 - val_mae: 40.6862 - learning_rate: 0.0010
Epoch 3/300
267/267 - 2s - 6ms/step - loss: 13988.3877 - mae: 39.0762 - val_loss: 17669.1914 - val_mae: 36.6404 - learning_rate: 0.0010
Epoch 4/300
267/267 - 2s - 6ms/step - loss: 13270.9980 - mae: 38.1627 - val_loss: 15936.3623 - val_mae: 35.3682 - learning_rate: 0.0010
Epoch 5/300
267/267 - 4s - 14ms/step - loss: 12632.9209 - mae: 37.8640 - val_loss: 14209.7686 - val_mae: 35.0427 - learning_rate: 0.0010
Epoch 6/300
267/267 - 2s - 7ms/step - loss: 12209.2803 - mae: 37.0863 - val_loss: 13109.7715 - val_mae: 36.4572 - learning_rate: 0.0010
Epoch 7/300
267/267 - 2s - 6ms/step - loss: 11996.5762 - mae: 37.0568 - val_loss: 11792.5898 - val_mae: 34.3801 - learning_rate: 0.0010
Epoch 8/300
267/267 - 2s - 6ms/step - loss: 11

In [47]:
y_pred = model.predict(X_test).ravel()

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [58]:
print(f"mar of test : {mae}")

mar of test : 37.46671315919172
